# Deep Reaserch

- Using
    - LangGraph
    - Ollama
    - Search tools (Google Serper, Tavily)


In [1]:
import os
import warnings
from pathlib import Path
from typing import Annotated, Any, Literal, TypedDict

# Standard imports
import numpy as np
import pandas as pd
import polars as pl

# Visualization
# import matplotlib.pyplot as plt

# NumPy settings
np.set_printoptions(precision=4)

# Pandas settings
pd.options.display.max_rows = 1_000
pd.options.display.max_columns = 1_000
pd.options.display.max_colwidth = 600

# Polars settings
pl.Config.set_fmt_str_lengths(1_000)
pl.Config.set_tbl_cols(n=1_000)
pl.Config.set_tbl_rows(n=200)

warnings.filterwarnings("ignore")

# Black code formatter (Optional)
%load_ext lab_black

# auto reload imports
%load_ext autoreload
%autoreload 2

In [2]:
from rich.console import Console
from rich.theme import Theme

custom_theme = Theme(
    {
        "white": "#FFFFFF",  # Bright white
        "info": "#00FF00",  # Bright green
        "warning": "#FFD700",  # Bright gold
        "error": "#FF1493",  # Deep pink
        "success": "#00FFFF",  # Cyan
        "highlight": "#FF4500",  # Orange-red
    }
)
console = Console(theme=custom_theme)


def create_path(path: str | Path) -> None:
    """
    Create parent directories for the given path if they don't exist.

    Parameters
    ----------
    path : str | Path
        The file path for which to create parent directories.
    """
    # Convert to Path object if it's a string
    path_obj: Path = Path(path) if isinstance(path, str) else path

    # Get the parent directory and create it if it doesn't exist
    path_obj.parent.mkdir(parents=True, exist_ok=True)


def go_up_from_current_directory(*, go_up: int = 1) -> None:
    """This is used to up a number of directories.

    Params:
    -------
    go_up: int, default=1
        This indicates the number of times to go back up from the current directory.

    Returns:
    --------
    None
    """
    import sys

    CONST: str = "../"
    NUM: str = CONST * go_up

    # Goto the previous directory
    prev_directory = os.path.join(os.path.dirname(__name__), NUM)
    # Get the 'absolute path' of the previous directory
    abs_path_prev_directory = os.path.abspath(prev_directory)

    # Add the path to the System paths
    sys.path.insert(0, abs_path_prev_directory)
    print(abs_path_prev_directory)

In [3]:
go_up_from_current_directory(go_up=2)

from model_config import LocalModel, RemoteModel  # noqa: E402
from settings import refresh_settings  # noqa: E402

settings = refresh_settings()

/Users/mac/Desktop/Projects/RAG-Tutorials


In [4]:
from langchain_openai import ChatOpenAI

model_str_remote: str = RemoteModel.QWEN3_30B_A3B
# model_str2_remote: str = RemoteModel.QWEN3_30B_A3B
model_str_local: str = LocalModel.QWEN3_4B_INSTRUCT_2507

remote_llm = ChatOpenAI(
    api_key=settings.OPENROUTER_API_KEY.get_secret_value(),  # type: ignore
    base_url=settings.OPENROUTER_URL,
    temperature=0.0,
    model=model_str_remote,
)


local_llm = ChatOpenAI(
    api_key=settings.LMSTUDIO_API_KEY.get_secret_value(),
    base_url=settings.LMSTUDIO_URL,
    temperature=0.0,
    model="model_str_local",
)

### States

In [5]:
import operator
from dataclasses import dataclass, field


@dataclass(kw_only=True)
class SummaryState:
    research_topic: str = field(default=None)
    search_query: str = field(default=None)
    web_research_results: Annotated[list, operator.add] = field(default_factory=list)
    sources_gathered: Annotated[list, operator.add] = field(default_factory=list)
    research_loop_count: int = field(default=0)  # Research loop count
    running_summary: str = field(default=None)  # Final report


@dataclass(kw_only=True)
class SummaryStateInput:
    research_topic: str = field(default=None)


@dataclass(kw_only=True)
class SummaryStateOutput:
    running_summary: str = field(default=None)

<br>

### Prompts

In [6]:
from datetime import datetime


# Get current date in a readable format
def get_current_date() -> str:
    """Get the current date in a readable format."""
    return datetime.now().strftime("%B %d, %Y")


query_writer_prompt: str = """
<GOAL>Your goal is to generate an optimized web search query based on the user's query</GOAL>

<CONTEXT>
Current date: {current_date}
Please ensure your queries account for the most current available information using the latest data available.
</CONTEXT>

<TOPIC> {research_topic} </TOPIC>

<EXAMPLE>
{{
    "query": "retrieval augmented generation (rag) explained simply",
    "rationale": "understanding the fundamental concept of retrieval augmented generation (rag)"
}}
</EXAMPLE>
"""

json_mode_query_prompt: str = """

<GOAL>
    Format your response as a JSON object with EXACTLY the following keys:
    - "query": The actual search query
    - "rationale": Brief explanation why this query is relevant
</GOAL>

<REQUIREMENTS>
    - Output MUST have the following keys:
      - "query"
      - "rationale"
</REQUIREMENTS>

<OUTPUT> Please provide your response in the specified JSON format: </OUTPUT>
"""

tool_calling_query_prompt: str = """

<GOAL>
    Use the `Query` tool with these required fields:
    - query: search terms
    - rationale: why this query helps
</GOAL>

<REQUIREMENTS>
    - Call the `Query` tool with the REQUIRED arguments
    - Output MUST have the following keys:
      - "query"
      - "rationale"
</REQUIREMENTS>

<OUTPUT>
    Please provide your response in the specified JSON format:
</OUTPUT>
"""

summarizer_prompt: str = """
<GOAL>
Generate a high-quality summary of the provided context.
</GOAL>

<REQUIREMENTS>
    When creating a NEW summary:
    - Highlight the most relevant information related to the user topic from the search results
    - Ensure a coherent flow of information

    When EXTENDING an existing summary:         
    - Read the existing summary and new search results carefully.
    - Compare the new information with the existing summary.     
    - For each piece of new information:                         
        a. If it's related to existing points, integrate it into the relevant paragraph.                               
        b. If it's entirely new but relevant, add a new paragraph with a smooth transition.                            
        c. If it's not relevant to the user topic, skip it.        
    - Ensure all additions are relevant to the user's topic.     
    - Verify that your final output differs from the input summary.
</REQUIREMENTS>

<FORMATTING>
    - Start directly with the updated summary, without preamble or titles. Do not use XML tags in the output.
</FORMATTING>

<TASK>
    Think carefully about the provided Context first. Then generate a summary of the context to address the User Input.
</TASK>
"""

reflection_prompt: str = """
<ROLE>
    You are an expert research assistant analyzing a summary about {research_topic}.
    </ROLE>

    <GOAL>
    - Identify knowledge gaps or areas that need deeper exploration
    - Generate a follow-up question that would help expand your understanding
    - Focus on technical details, implementation specifics, or emerging trends that weren't fully covered
</GOAL>

<REQUIREMENTS>
    Ensure the follow-up question is self-contained and includes necessary context for web search.
</REQUIREMENTS>
"""

json_mode_reflection_prompt: str = """
<FORMAT>
    Format your response as a JSON object with **EXACTLY** the following keys:
    - knowledge_gap: Describe what information is missing or needs clarification
    - follow_up_query: Write a specific question to address this gap
</FORMAT>

<TASK>
    - Reflect carefully on the summary to identify knowledge gaps and produce a follow-up query. Then, produce 
    your output following this JSON format:
    {{
        "knowledge_gap": "The summary lacks information about performance metrics and benchmarks",
        "follow_up_query": "What are typical performance benchmarks and metrics used to evaluate [specific technology]?"
    }}
</TASK>

Provide your analysis in JSON format:
"""

tool_calling_reflection_prompt: str = """
<INSTRUCTIONS>
    Call the `FollowUpQuery` tool to format your response with **EXACTLY** the following keys:
    - follow_up_query: Write a specific question to address this gap
    - knowledge_gap: Describe what information is missing or needs clarification
</INSTRUCTIONS>

<TASK>
    Reflect carefully on the Summary to identify knowledge gaps and produce a follow-up query.
</TASK>

<REQUIREMENTS>
    - Call the `FollowUpQuery` tool with the REQUIRED arguments
    - Output must match have the following keys:
      - "follow_up_query"
      - "knowledge_gap"
</REQUIREMENTS>

<OUTPUT>
    Please provide your response in the specified JSON format:
</OUTPUT>
"""

### Configurations

In [21]:
from enum import Enum

from langchain_core.runnables import RunnableConfig
from pydantic import BaseModel, Field


class SearchAPI(Enum):
    PERPLEXITY = "perplexity"
    TAVILY = "tavily"
    DUCKDUCKGO = "duckduckgo"
    SEARXNG = "searxng"


class Configuration(BaseModel):
    """The configurable fields for the research assistant."""

    max_web_research_loops: int = Field(
        default=3,
        title="Research Depth",
        description="Number of research iterations to perform",
    )
    local_llm: str = Field(
        default=LocalModel.QWEN3_4B_INSTRUCT_2507,
        title="LLM Model Name",
        description="Name of the LLM model to use",
    )
    llm_provider: Literal["ollama", "lmstudio"] = Field(
        default="lmstudio",
        title="LLM Provider",
        description="Provider for the LLM (Ollama or LMStudio)",
    )
    search_api: Literal["tavily", "searxng", "google_serper", "exa"] = Field(
        default="searxng", title="Search API", description="Web search API to use"
    )
    fetch_full_page: bool = Field(
        default=True,
        title="Fetch Full Page",
        description="Include the full page content in the search results",
    )
    ollama_base_url: str = Field(
        default="http://localhost:11434/",
        title="Ollama Base URL",
        description="Base URL for Ollama API",
    )
    lmstudio_base_url: str = Field(
        default="http://localhost:1234/v1",
        title="LMStudio Base URL",
        description="Base URL for LMStudio OpenAI-compatible API",
    )
    strip_thinking_tokens: bool = Field(
        default=True,
        title="Strip Thinking Tokens",
        description="Whether to strip <think> tokens from model responses",
    )
    use_tool_calling: bool = Field(
        default=False,
        title="Use Tool Calling",
        description="Use tool calling instead of JSON mode for structured output",
    )

    @classmethod
    def from_runnable_config(
        cls, config: RunnableConfig | None = None
    ) -> "Configuration":
        """Create a Configuration instance from a RunnableConfig."""
        configurable = (
            config["configurable"] if config and "configurable" in config else {}
        )

        # Get raw values from environment or config
        raw_values: dict[str, Any] = {
            name: os.environ.get(name.upper(), configurable.get(name))
            for name in cls.model_fields.keys()
        }

        # Filter out None values
        values = {k: v for k, v in raw_values.items() if v is not None}

        return cls(**values)

In [8]:
configuration: Configuration = Configuration.from_runnable_config()
configuration.model_dump()

{'max_web_research_loops': 3,
 'local_llm': <LocalModel.QWEN3_4B_INSTRUCT_2507: 'qwen3-4b-instruct-2507'>,
 'llm_provider': 'lmstudio',
 'search_api': 'searxng',
 'fetch_full_page': True,
 'ollama_base_url': 'http://localhost:11434/',
 'lmstudio_base_url': 'http://localhost:1234/v1',
 'strip_thinking_tokens': True,
 'use_tool_calling': False}

### Utils

In [9]:
import json


# utils.py
def strip_thinking_tokens(text: str) -> str:
    """
    Remove <think> and </think> tags and their content from the text.

    Iteratively removes all occurrences of content enclosed in thinking tokens.

    Args:
        text (str): The text to process

    Returns:
        str: The text with thinking tokens and their content removed
    """
    while "<think>" in text and "</think>" in text:
        start: int = text.find("<think>")
        end: int = text.find("</think>") + len("</think>")
        text = text[:start] + text[end:]
    return text


def get_llm(configurable: Configuration) -> ChatOpenAI:
    """Helper function to initialize LLM based on configuration.

    Uses JSON mode if use_tool_calling is False, otherwise regular mode for tool calling.

    Args:
        configurable: Configuration object containing LLM settings

    Returns:
        Configured LLM instance
    """
    if configurable.llm_provider == "lmstudio":
        if configurable.use_tool_calling:
            return ChatOpenAI(
                base_url=configurable.lmstudio_base_url,
                model=configurable.local_llm,
                temperature=0,
            )
        return ChatOpenAI(
            base_url=configurable.lmstudio_base_url,
            model=configurable.local_llm,
            temperature=0,
        )
    # Default to Ollama
    if configurable.use_tool_calling:
        return ChatOpenAI(
            base_url=configurable.ollama_base_url,
            model=configurable.local_llm,
            temperature=0,
        )
    return ChatOpenAI(
        base_url=configurable.ollama_base_url,
        model=configurable.local_llm,
        temperature=0,
    )

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool


@tool
def add(a: float, b: float) -> float:
    """Add two numbers together."""
    return a + b


@tool
class Query:
    """A  tool for generating queries."""

    query: str = Field(description="The actual search query string")
    rationale: str = Field(description="Brief explanation of why this query is relevant")


query = query_writer_prompt.format(current_date=get_current_date(), research_topic="explain fantasy premier league")
use_tool_calling = True
messages = [
    SystemMessage(content=query + (tool_calling_query_prompt if use_tool_calling else json_mode_query_prompt)),
    HumanMessage(content="Generate a query for web search:"),
]
print(f"Original query: {messages[0].content}")
response = local_llm.bind_tools([add, Query]).invoke(messages)
console.print(response)

Original query: 
<GOAL>Your goal is to generate an optimized web search query based on the user's query</GOAL>

<CONTEXT>
Current date: August 18, 2025
Please ensure your queries account for the most current available information using the latest data available.
</CONTEXT>

<TOPIC> explain fantasy premier league </TOPIC>

<EXAMPLE>
{
    "query": "retrieval augmented generation (rag) explained simply",
    "rationale": "understanding the fundamental concept of retrieval augmented generation (rag)"
}
</EXAMPLE>


<GOAL>
    Use the `Query` tool with these required fields:
    - query: search terms
    - rationale: why this query helps
</GOAL>

<REQUIREMENTS>
    - Call the `Query` tool with the REQUIRED arguments
    - Output MUST have the following keys:
      - "query"
      - "rationale"
</REQUIREMENTS>

<OUTPUT>
    Please provide your response in the specified JSON format:
</OUTPUT>



AIMessage(
    content='',
    additional_kwargs={
        'tool_calls': [
            {
                'id': '778865492',
                'function': {
                    'arguments': '{"query":"explain fantasy premier league 2025 rules, how it works, team 
management tips","rationale":"This query targets the most current information about Fantasy Premier League as of 
August 18, 2025, including updated rules, gameplay mechanics, and strategic advice for players in the latest 
season."}',
                    'name': 'Query'
                },
                'type': 'function'
            }
        ],
        'refusal': None
    },
    response_metadata={
        'token_usage': {
            'completion_tokens': 83,
            'prompt_tokens': 423,
            'total_tokens': 506,
            'completion_tokens_details': None,
            'prompt_tokens_details': None
        },
        'model_name': 'qwen3-4b-instruct-2507',
        'system_fingerprint': 'qwen3-4b-instruct-2507',
        'id': 'chatcmpl-jg9uxtd1wcpeexg3ly04e',
        'service_tier': None,
        'finish_reason': 'tool_calls',
        'logprobs': None
    },
    id='run--ff624126-e544-4068-95d9-511fa933ce30-0',
    tool_calls=[
        {
            'name': 'Query',
            'args': {
                'query': 'explain fantasy premier league 2025 rules, how it works, team management tips',
                'rationale': 'This query targets the most current information about Fantasy Premier League as of 
August 18, 2025, including updated rules, gameplay mechanics, and strategic advice for players in the latest 
season.'
            },
            'id': '778865492',
            'type': 'tool_call'
        }
    ],
    usage_metadata={
        'input_tokens': 423,
        'output_tokens': 83,
        'total_tokens': 506,
        'input_token_details': {},
        'output_token_details': {}
    }
)

In [ ]:
# Local models work best when used with JSON mode instead of TOOL mode
use_tool_calling = False
messages = [
    SystemMessage(content=query + (tool_calling_query_prompt if use_tool_calling else json_mode_query_prompt)),
    HumanMessage(content="Generate a query for web search:"),
]
print(f"Original query: {messages[0].content}")
response = local_llm.invoke(messages)
console.print(response.content)
search_query: str = json.loads(response.content)["query"]
print(f"Search query: {search_query}")

Original query: 
<GOAL>Your goal is to generate an optimized web search query based on the user's query</GOAL>

<CONTEXT>
Current date: August 18, 2025
Please ensure your queries account for the most current available information using the latest data available.
</CONTEXT>

<TOPIC> explain fantasy premier league </TOPIC>

<EXAMPLE>
{
    "query": "retrieval augmented generation (rag) explained simply",
    "rationale": "understanding the fundamental concept of retrieval augmented generation (rag)"
}
</EXAMPLE>


<GOAL>
    Format your response as a JSON object with EXACTLY the following keys:
    - "query": The actual search query
    - "rationale": Brief explanation why this query is relevant
</GOAL>

<REQUIREMENTS>
    - Output MUST have the following keys:
      - "query"
      - "rationale"
</REQUIREMENTS>

<OUTPUT> Please provide your response in the specified JSON format: </OUTPUT>



{
    "query": "fantasy premier league explained for beginners 2025",
    "rationale": "This query targets a clear, beginner-friendly explanation of Fantasy Premier League, 
incorporating the most current year (2025) to ensure up-to-date information on rules, formats, and features 
relevant to users starting in the league."
}

Search query: fantasy premier league explained for beginners 2025


### Nodes

In [ ]:
from typing import Type, TypeVar

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage

T = TypeVar("T", bound=BaseModel)


def generate_search_query_with_structured_output(
    configurable: Configuration,
    messages: list,
    tool_class: Type[T],
    fallback_query: str,
    tool_query_field: str,
    json_query_field: str,
) -> dict[str, Any]:
    """Helper function to generate search queries using either tool calling or JSON mode.

    Args:
        configurable: Configuration object
        messages: List of messages to send to LLM
        tool_class: Tool class for tool calling mode
        fallback_query: Fallback search query if extraction fails
        tool_query_field: Field name in tool args containing the query
        json_query_field: Field name in JSON response containing the query

    Returns:
        Dictionary with "search_query" key
    """
    if configurable.use_tool_calling:
        llm: ChatOpenAI = get_llm(configurable).bind_tools([tool_class])
        result = llm.invoke(messages)

        if not result.tool_calls:
            # If no tool calls were made
            return {"search_query": fallback_query}

        try:
            tool_data = result.tool_calls[0]["args"]
            search_query = tool_data.get(tool_query_field)
            return {"search_query": search_query}

        except (IndexError, KeyError):
            return {"search_query": fallback_query}

    else:
        # Use JSON mode
        llm = get_llm(configurable)
        result = llm.invoke(messages)
        print(f"result: {result}")
        content = result.content

        try:
            parsed_json = json.loads(content)
            search_query = parsed_json.get(json_query_field)
            if not search_query:
                return {"search_query": fallback_query}
            return {"search_query": search_query}

        except (json.JSONDecodeError, KeyError):
            if configurable.strip_thinking_tokens:
                content = strip_thinking_tokens(content)
            return {"search_query": fallback_query}


def generate_query(state: SummaryState, config: RunnableConfig) -> dict[str, Any]:
    """LangGraph node that generates a search query based on the research topic.

    Uses an LLM to create an optimized search query for web research based on
    the user's research topic. Supports both LMStudio and Ollama as LLM providers.

    Args:
        state: Current graph state containing the research topic
        config: Configuration for the runnable, including LLM provider settings

    Returns:
        Dictionary with state update, including search_query key containing the generated query
    """

    # Format the prompt
    current_date: str = get_current_date()
    formatted_prompt = query_writer_prompt.format(current_date=current_date, research_topic=state.research_topic)

    # Generate a query
    configurable = Configuration.from_runnable_config(config)

    @tool
    class Query(BaseModel):
        """
        This tool is used to generate a query for web search.
        """

        query: str = Field(description="The actual search query string")
        rationale: str = Field(description="Brief explanation of why this query is relevant")

    messages = [
        SystemMessage(
            content=formatted_prompt
            + (tool_calling_query_prompt if configurable.use_tool_calling else json_mode_query_prompt)
        ),
        HumanMessage(content="Generate a query for web search:"),
    ]

    return generate_search_query_with_structured_output(
        configurable=configurable,
        messages=messages,
        tool_class=Query,
        fallback_query=f"Tell me more about {state.research_topic}",
        tool_query_field="query",
        json_query_field="query",
    )

In [22]:
import httpx
from langchain_community.utilities import GoogleSerperAPIWrapper, SearxSearchWrapper
from langchain_exa import ExaSearchResults
from langchain_tavily import TavilySearch
from markdownify import markdownify


def fetch_raw_content(url: str) -> str | None:
    """
    Fetch HTML content from a URL and convert it to markdown format.

    Parameters
    ----------
    url : str
        The URL to fetch content from.

    Returns
    -------
    str or None
        The fetched content converted to markdown if successful,
        None if any error occurs during fetching or conversion.

    Notes
    -----
    Uses a 10-second timeout to avoid hanging on slow sites or large pages.
    """
    try:
        # Create a client with reasonable timeout
        with httpx.Client(timeout=10.0) as client:
            response = client.get(url)
            response.raise_for_status()
            return markdownify(response.text)
    except Exception as e:
        print(f"Warning: Failed to fetch full page content for {url}: {str(e)}")
        return None


def searxng_search(
    query: str, max_results: int = 3, fetch_full_page: bool = False
) -> dict[str, list[dict[str, Any]]]:
    """
    Search the web using SearXNG with improved error handling.

    Parameters
    ----------
    query : str
        The search query string.
    max_results : int, optional
        Maximum number of results to return (default is 3).
    fetch_full_page : bool, optional
        If True, fetch and include the full page content for each result (default is False).

    Returns
    -------
    dict[str, list[dict[str, Any]]]
        Dictionary with a "results" key containing a list of result dictionaries.
        Each result dictionary contains:
            - "title": str, the result title
            - "url": str, the result URL
            - "content": str, the snippet or summary
            - "raw_content": str, the full page content if fetched, otherwise the snippet

    Notes
    -----
    Uses the SearxSearchWrapper for querying SearXNG. Handles incomplete results and fetch errors gracefully.
    """
    host = os.environ.get("SEARXNG_URL", "http://localhost:8080")

    try:
        s = SearxSearchWrapper(searx_host=host)
        results = []
        search_results = s.results(query, num_results=max_results)

        for r in search_results:
            url = r.get("link")
            title = r.get("title")
            content = r.get("snippet", "")

            if not all([url, title]):
                print(f"Warning: Incomplete result from SearXNG: {r}")
                continue

            raw_content = content
            if fetch_full_page:
                raw_content = fetch_raw_content(url)
                if raw_content is None:
                    raw_content = content

            result = {
                "title": title,
                "url": url,
                "content": content,
                "raw_content": raw_content,
            }
            results.append(result)

        return {"results": results}

    except Exception as e:
        print(f"SearXNG search failed: {str(e)}")
        return {"results": []}


def exa_search(query: str, num_results: int = 3) -> dict[str, list[dict[str, Any]]]:
    """
    Perform a web search using the ExaSearchResults tool.

    Parameters
    ----------
    query : str
        The search query string.
    num_results : int, optional
        Maximum number of results to return (default is 3).

    Returns
    -------
    dict[str, list[dict[str, Any]]]
        Dictionary with a "results" key containing a list of result dictionaries.
        Each result dictionary contains:
            - "title": str, the result title
            - "url": str, the result URL
            - "content": str, truncated text content
            - "raw_content": str, the full text content

    Notes
    -----
    Uses the ExaSearchResults tool for querying Exa. Handles errors gracefully and truncates content to a character limit.
    """
    character_limit: int = 1_500
    # Initialize the ExaSearchResults tool
    search_tool = ExaSearchResults(exa_api_key=settings.EXA_API_KEY.get_secret_value())

    try:
        # Perform a search query
        search_results = search_tool._run(  # noqa: SLF001
            query=query,
            num_results=num_results,
            text_contents_options=True,
            highlights=True,
        )
        results: list[dict[str, Any]] = [
            {
                "title": result.title,
                "url": result.url,
                "content": result.text[:character_limit] + "... </truncated>",
                "raw_content": result.text,
            }
            for result in search_results.results
        ]
        return {"results": results}

    except Exception as e:
        print(f"Exa search failed: {str(e)}")
        return {"results": []}


def tavily_search(query: str, max_results: int = 3) -> dict[str, list[dict[str, Any]]]:
    """
    Perform a web search using the TavilySearch tool.

    Parameters
    ----------
    query : str
        The search query string.
    max_results : int, optional
        Maximum number of results to return (default is 3).

    Returns
    -------
    dict[str, list[dict[str, Any]]]
        Dictionary with a "results" key containing a list of result dictionaries.
        Each result dictionary contains:
            - "title": str, the result title
            - "url": str, the result URL
            - "content": str, the snippet or summary
            - "raw_content": str, the full page content if fetched, otherwise the snippet

    Notes
    -----
    Uses the TavilySearch tool for querying Tavily. Handles errors gracefully.
    """
    search_tool = TavilySearch(max_results=max_results, topic="general")
    try:
        results: list[dict[str, Any]] = search_tool.invoke(query)["results"]
        return {"results": results}

    except Exception as e:
        print(f"Tavily search failed: {str(e)}")
    return {"results": []}


def google_search(
    query: str, max_results: int = 3, fetch_full_page: bool = False
) -> dict[str, list[dict[str, Any]]]:
    """
    Perform a web search using the GoogleSerperAPIWrapper.

    Parameters
    ----------
    query : str
        The search query string.
    max_results : int, optional
        Maximum number of results to return (default is 3).
    fetch_full_page : bool, optional
        If True, fetch and include the full page content for each result (default is False).

    Returns
    -------
    dict[str, list[dict[str, Any]]]
        Dictionary with a "results" key containing a list of result dictionaries.
        Each result dictionary contains:
            - "title": str, the result title
            - "url": str, the result URL
            - "content": str, the snippet or summary
            - "raw_content": str, the full page content if fetched, otherwise the snippet

    Notes
    -----
    Uses the GoogleSerperAPIWrapper for querying Google Serper. Handles errors gracefully.
    """
    search = GoogleSerperAPIWrapper(k=max_results)

    try:
        raw_results = search.results(query)

        results = [
            {
                "title": res["title"],
                "url": res["link"],
                "content": res["snippet"],
                "raw_content": res["snippet"],
            }
            for res in raw_results["organic"]
        ]
        if fetch_full_page:
            for res in results:
                res["raw_content"] = fetch_raw_content(res["url"])
        return {"results": results}

    except Exception as e:
        print(f"Google search failed: {str(e)}")
        return {"results": []}

In [23]:
query = "What was the score of Chelsea vs Crystal Palace?"
search_results = searxng_search(query)
console.print("Search Results:", search_results)

SearXNG search failed: HTTPConnectionPool(host='localhost', port=8080): Max retries exceeded with url: /?language=en&format=json&q=What+was+the+score+of+Chelsea+vs+Crystal+Palace%3F (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x140087ec0>: Failed to establish a new connection: [Errno 61] Connection refused'))


Search Results:
{'results': []}

In [ ]:
search_results.results[0].url
# search_results.results[0].text[:1_500] + "... </truncated>"  # content
# search_results.results[0].text  # raw_content

In [ ]:
res = searxng_search(
    "What was the score of Chelsea vs Crystal Palace?",
    fetch_full_page=True,
)
console.print(res)

In [ ]:
res["results"][0]["raw_content"]

# Using LangGraph

### Agent Workflow

- A user submits a query.

- The agent determines if a tool is needed or if its internal knowledge is sufficient.

- If a tool is required, the agent executes it with the necessary information, and the result is sent back for the agent to process.

- If no tool is needed, the agent generates a response based on its own knowledge and the user's input.

<br>

[![image.png](https://i.postimg.cc/wMRfT7vF/image.png)](https://postimg.cc/ctZMF1f8)

In [ ]:
from langgraph.graph.message import add_messages


class AgentState(TypedDict):
    messages: Annotated[list[Any], add_messages]


@tool
def add(a: float, b: float) -> float:
    """Add two numbers."""
    return a + b


@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers."""
    return a * b


@tool
def local_wiki(query: str) -> str:
    """Search the local wiki for a query."""
    response: str = """
    Emmanuel Obi is a 28-year-old AI Engineer based in Enugu, Nigeria. He has a keen professional interest in 
    Natural Language Processing (NLP). When he's not working, Emmanuel enjoys a variety of hobbies. He's a big 
    fan of football and a dedicated supporter of his favorite team, Chelsea FC. He also finds time to stay 
    active by working out and unwinds by listening to music. As a young professional, Emmanuel embodies a blend 
    of technical expertise and a vibrant personal life.
    """
    return response


tools = [add, multiply, local_wiki]
llm_with_tools = remote_llm_with_tools.bind_tools(tools)

In [ ]:
sys_msg: str = """
<system>
    <role>
    You're a helpful assistant that answers user questions in a concise and informative manner.
    You have access to a variety of tools and resources to assist with answering questions.
    </role>

    <guidlines>
    - Be concise and informative in your responses.
    - Use the available tools and resources to enhance your answers.
    - Maintain a friendly and helpful tone.
    </guidlines>
</system>
"""


system_message = SystemMessage(content=sys_msg)


# Node
def assistant(state: AgentState) -> dict[str, Any]:
    return {"messages": [llm_with_tools.invoke([system_message] + state["messages"])]}


def route_tools(state: AgentState) -> Literal["tools", "__end__"]:
    # Check if state is a list (legacy or alternate format)
    if isinstance(state, list):
        ai_message = state[-1]  # Get the last message (assumed to be AIMessage)
    # Otherwise, try to get messages from the state dictionary
    elif messages := state.get("messages", []):
        ai_message = messages[-1]  # Get the last message (assumed to be AIMessage)
    else:
        # If no messages are found, raise an error for debugging
        raise ValueError(f"No message found in the input state to tool_edge: {state}")

    # If the AI message contains tool calls, route to the "tools" node
    if hasattr(ai_message, "tool_calls") and len(ai_message.tool_calls) > 0:
        return "tools"
    # Otherwise, end the workflow
    return "__end__"

In [ ]:
from langgraph.graph import START, END, StateGraph
from langgraph.prebuilt import tools_condition
from langgraph.prebuilt import ToolNode
from IPython.display import Image, display


# Graph
builder = StateGraph(AgentState)

# Nodes
builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode(tools=tools))

# Edges
builder.add_edge(START, "assistant")
# If the assistant needs to use a tool, route to the tools node
builder.add_conditional_edges("assistant", route_tools)
# If a tool was used, route back to the assistant
builder.add_edge("tools", "assistant")

# Build
react_graph = builder.compile()
# Visualize the graph
display(Image(react_graph.get_graph(xray=1).draw_mermaid_png()))

In [ ]:
query: str = "What is the sum of 3 and 9? Multiply the result by 2.5."
messages = [HumanMessage(content=query)]
responses = react_graph.invoke({"messages": messages})

In [ ]:
for m in responses["messages"]:
    m.pretty_print()

In [ ]:
query: str = "Who is Emmanuel Obi? How old is he now and how old will he be in 5 years?"
messages = [HumanMessage(content=query)]
responses = react_graph.invoke({"messages": messages})


for m in responses["messages"]:
    m.pretty_print()

In [ ]:
query: str = "What was my first question?"
messages = [HumanMessage(content=query)]
responses = react_graph.invoke({"messages": messages})


for m in responses["messages"]:
    m.pretty_print()

### Add Memory

```py
from langgraph.checkpoint.memory import MemorySaver
```

In [ ]:
from langgraph.checkpoint.memory import MemorySaver


# Graph
builder = StateGraph(AgentState)

# Nodes
builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode(tools=tools))

# Edges
builder.add_edge(START, "assistant")
# If the assistant needs to use a tool, route to the tools node
builder.add_conditional_edges("assistant", route_tools)
# If a tool was used, route back to the assistant
builder.add_edge("tools", "assistant")
builder.add_edge("assistant", END)

# Build
memory = MemorySaver()
react_graph = builder.compile(checkpointer=memory)

# Visualize the graph
display(Image(react_graph.get_graph(xray=1).draw_mermaid_png()))

In [ ]:
config = {"configurable": {"thread_id": "1"}}

# 1
query: str = "Hello, I'm Obi Emmanuel. What is the sum of 3 and 9? Multiply the result by 2.5."
# query: str = "Hello, I'm Obi Emmanuel"
messages = [HumanMessage(content=query)]
responses = react_graph.invoke({"messages": messages}, config=config)

for m in responses["messages"]:
    m.pretty_print()

In [ ]:
# 2
query: str = "What is my name?"

messages = [HumanMessage(content=query)]
responses = react_graph.invoke({"messages": messages}, config=config)

for m in responses["messages"]:
    m.pretty_print()

In [ ]:
# View the State
my_state = react_graph.get_state(config=config)
console.print(my_state)

### Practicing Conditional Logic

In [ ]:
class AgentStateWithSummarizer(TypedDict):
    messages: Annotated[list[Any], add_messages]
    summary: str

In [ ]:
from langchain_core.messages import RemoveMessage


generate_summary_sys_msg: str = """
<system>

    <role>
    You are a highly attentive assistant tasked with generating or extending summaries of conversations.
    Your goal is to retain all critical information, including names, entities, facts, and context, while being concise.
    </role>

    <previous_summary>
    {summary}
    </previous_summary>

    <instruction>
    - Carefully review the following messages and update the summary to include all important details, especially names, 
    entities, numbers, and any facts or context that may be relevant for future conversation. 
    - Do NOT omit any information that could help the assistant remember the user's identity, preferences, 
    or prior questions.
    - If the previous summary already contains some information, only add new details from the latest messages.
    </instruction>

    <new_messages>{messages}</new_messages>

    <guidelines>
    - Always preserve names, entities, numbers, and facts.
    - Do not generalize or omit specifics.
    - Make the summary concise but comprehensive.
    - The summary should help the assistant recall the user's context and history.
    </guidelines>
    
</system>
"""

sys_msg_with_context: str = """
<system>

    <role>
    You're a helpful assistant that answers user questions in a concise and informative manner considering the context.
    </role>

    <context> \n\n{context} </context>

    <guidlines>
    - Be concise and informative in your responses.
    - Maintain a friendly and helpful tone.
    </guidlines>

</system>
"""


# Nodes
def chatbot(state: AgentStateWithSummarizer) -> dict[str, Any]:
    summary: str = state.get("summary", "")
    context = sys_msg_with_context.format(context=summary)
    response = remote_llm.invoke([SystemMessage(content=context)] + state["messages"])
    print(f"Summary: {state.get('summary', '')}")
    return {"messages": response}


def summarize_messages(state: AgentStateWithSummarizer) -> dict[str, Any]:
    summary: str = state.get("summary", "")
    sys_msg_summarizer = generate_summary_sys_msg.format(summary=summary, messages=state["messages"])
    response = remote_llm.invoke([SystemMessage(content=sys_msg_summarizer)])
    # Delete all but the last 2 messages
    deleted_msgs = [RemoveMessage(id=m.id) for m in state["messages"][:-2]]
    return {"summary": response.content, "messages": deleted_msgs}


# Conditional Node
def should_continue(
    state: AgentStateWithSummarizer,
) -> Literal["summarize_messages", "__end__"]:
    if len(state["messages"]) > 5:
        return "summarize_messages"
    return "__end__"

In [ ]:
# Graph
builder = StateGraph(AgentState)

# Nodes
builder.add_node("chatbot", chatbot)
builder.add_node("summarize_messages", summarize_messages)

# Edges
builder.add_edge(START, "chatbot")
# If the chatbot needs to use a tool, route to the tools node
builder.add_conditional_edges("chatbot", should_continue)
# If a tool was used, route back to the chatbot
builder.add_edge("chatbot", END)

# Build
memory = MemorySaver()
simple_graph = builder.compile(checkpointer=memory)

# Visualize the graph
display(Image(simple_graph.get_graph(xray=1).draw_mermaid_png()))

In [ ]:
config = {"configurable": {"thread_id": "1"}}

# 1
messages: list = [
    HumanMessage(content="Hi, I'm, Neidu"),
    AIMessage(content="Hello, Neidu! How can I assist you today?"),
    HumanMessage(content="What is the weather like today?"),
    AIMessage(content="The weather is sunny with a high of 25°C."),
    HumanMessage(content="Thank you!"),
    AIMessage(content="You're welcome! If you have any more questions, feel free to ask."),
    HumanMessage(content="What do you know about AI agents?"),
]
responses = simple_graph.invoke({"messages": messages}, config=config)

for m in responses["messages"]:
    m.pretty_print()

In [ ]:
query: str = "What do you know about me so far? What's my name?"
messages = [HumanMessage(content=query)]
responses = simple_graph.invoke({"messages": messages}, config=config)

for m in responses["messages"]:
    m.pretty_print()

In [ ]:
query: str = "I'm sure I mentioned my name earlier."
messages = [HumanMessage(content=query)]
responses = simple_graph.invoke({"messages": messages}, config=config)

for m in responses["messages"]:
    m.pretty_print()

In [ ]:
import json

from pydantic import BaseModel, Field
from typing_extensions import Literal

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.runnables import RunnableConfig
from langchain_core.tools import tool
from langchain_ollama import ChatOllama
from langgraph.graph import START, END, StateGraph

from ollama_deep_researcher.configuration import Configuration, SearchAPI
from ollama_deep_researcher.utils import (
    deduplicate_and_format_sources,
    tavily_search,
    format_sources,
    perplexity_search,
    duckduckgo_search,
    searxng_search,
    strip_thinking_tokens,
    get_config_value,
)
from ollama_deep_researcher.state import (
    SummaryState,
    SummaryStateInput,
    SummaryStateOutput,
)
from ollama_deep_researcher.prompts import (
    query_writer_instructions,
    summarizer_instructions,
    reflection_instructions,
    get_current_date,
    json_mode_query_instructions,
    tool_calling_query_instructions,
    json_mode_reflection_instructions,
    tool_calling_reflection_instructions,
)
from ollama_deep_researcher.lmstudio import ChatLMStudio

# Constants
MAX_TOKENS_PER_SOURCE = 1000
CHARS_PER_TOKEN = 4


def generate_search_query_with_structured_output(
    configurable: Configuration,
    messages: list,
    tool_class,
    fallback_query: str,
    tool_query_field: str,
    json_query_field: str,
):
    """Helper function to generate search queries using either tool calling or JSON mode.

    Args:
        configurable: Configuration object
        messages: List of messages to send to LLM
        tool_class: Tool class for tool calling mode
        fallback_query: Fallback search query if extraction fails
        tool_query_field: Field name in tool args containing the query
        json_query_field: Field name in JSON response containing the query

    Returns:
        Dictionary with "search_query" key
    """
    if configurable.use_tool_calling:
        llm = get_llm(configurable).bind_tools([tool_class])
        result = llm.invoke(messages)

        if not result.tool_calls:
            return {"search_query": fallback_query}

        try:
            tool_data = result.tool_calls[0]["args"]
            search_query = tool_data.get(tool_query_field)
            return {"search_query": search_query}
        except (IndexError, KeyError):
            return {"search_query": fallback_query}

    else:
        # Use JSON mode
        llm = get_llm(configurable)
        result = llm.invoke(messages)
        print(f"result: {result}")
        content = result.content

        try:
            parsed_json = json.loads(content)
            search_query = parsed_json.get(json_query_field)
            if not search_query:
                return {"search_query": fallback_query}
            return {"search_query": search_query}
        except (json.JSONDecodeError, KeyError):
            if configurable.strip_thinking_tokens:
                content = strip_thinking_tokens(content)
            return {"search_query": fallback_query}


def get_llm(configurable: Configuration):
    """Helper function to initialize LLM based on configuration.

    Uses JSON mode if use_tool_calling is False, otherwise regular mode for tool calling.

    Args:
        configurable: Configuration object containing LLM settings

    Returns:
        Configured LLM instance
    """
    if configurable.llm_provider == "lmstudio":
        if configurable.use_tool_calling:
            return ChatLMStudio(
                base_url=configurable.lmstudio_base_url,
                model=configurable.local_llm,
                temperature=0,
            )
        else:
            return ChatLMStudio(
                base_url=configurable.lmstudio_base_url,
                model=configurable.local_llm,
                temperature=0,
                format="json",
            )
    else:  # Default to Ollama
        if configurable.use_tool_calling:
            return ChatOllama(
                base_url=configurable.ollama_base_url,
                model=configurable.local_llm,
                temperature=0,
            )
        else:
            return ChatOllama(
                base_url=configurable.ollama_base_url,
                model=configurable.local_llm,
                temperature=0,
                format="json",
            )


# Nodes
def generate_query(state: SummaryState, config: RunnableConfig):
    """LangGraph node that generates a search query based on the research topic.

    Uses an LLM to create an optimized search query for web research based on
    the user's research topic. Supports both LMStudio and Ollama as LLM providers.

    Args:
        state: Current graph state containing the research topic
        config: Configuration for the runnable, including LLM provider settings

    Returns:
        Dictionary with state update, including search_query key containing the generated query
    """

    # Format the prompt
    current_date = get_current_date()
    formatted_prompt = query_writer_instructions.format(current_date=current_date, research_topic=state.research_topic)

    # Generate a query
    configurable = Configuration.from_runnable_config(config)

    @tool
    class Query(BaseModel):
        """
        This tool is used to generate a query for web search.
        """

        query: str = Field(description="The actual search query string")
        rationale: str = Field(description="Brief explanation of why this query is relevant")

    messages = [
        SystemMessage(
            content=formatted_prompt
            + (tool_calling_query_instructions if configurable.use_tool_calling else json_mode_query_instructions)
        ),
        HumanMessage(content="Generate a query for web search:"),
    ]

    return generate_search_query_with_structured_output(
        configurable=configurable,
        messages=messages,
        tool_class=Query,
        fallback_query=f"Tell me more about {state.research_topic}",
        tool_query_field="query",
        json_query_field="query",
    )


def web_research(state: SummaryState, config: RunnableConfig):
    """LangGraph node that performs web research using the generated search query.

    Executes a web search using the configured search API (tavily, perplexity,
    duckduckgo, or searxng) and formats the results for further processing.

    Args:
        state: Current graph state containing the search query and research loop count
        config: Configuration for the runnable, including search API settings

    Returns:
        Dictionary with state update, including sources_gathered, research_loop_count, and web_research_results
    """

    # Configure
    configurable = Configuration.from_runnable_config(config)

    # Get the search API
    search_api = get_config_value(configurable.search_api)

    # Search the web
    if search_api == "tavily":
        search_results = tavily_search(
            state.search_query,
            fetch_full_page=configurable.fetch_full_page,
            max_results=1,
        )
        search_str = deduplicate_and_format_sources(
            search_results,
            max_tokens_per_source=MAX_TOKENS_PER_SOURCE,
            fetch_full_page=configurable.fetch_full_page,
        )
    elif search_api == "perplexity":
        search_results = perplexity_search(state.search_query, state.research_loop_count)
        search_str = deduplicate_and_format_sources(
            search_results,
            max_tokens_per_source=MAX_TOKENS_PER_SOURCE,
            fetch_full_page=configurable.fetch_full_page,
        )
    elif search_api == "duckduckgo":
        search_results = duckduckgo_search(
            state.search_query,
            max_results=3,
            fetch_full_page=configurable.fetch_full_page,
        )
        search_str = deduplicate_and_format_sources(
            search_results,
            max_tokens_per_source=MAX_TOKENS_PER_SOURCE,
            fetch_full_page=configurable.fetch_full_page,
        )
    elif search_api == "searxng":
        search_results = searxng_search(
            state.search_query,
            max_results=3,
            fetch_full_page=configurable.fetch_full_page,
        )
        search_str = deduplicate_and_format_sources(
            search_results,
            max_tokens_per_source=MAX_TOKENS_PER_SOURCE,
            fetch_full_page=configurable.fetch_full_page,
        )
    else:
        raise ValueError(f"Unsupported search API: {configurable.search_api}")

    return {
        "sources_gathered": [format_sources(search_results)],
        "research_loop_count": state.research_loop_count + 1,
        "web_research_results": [search_str],
    }


def summarize_sources(state: SummaryState, config: RunnableConfig):
    """LangGraph node that summarizes web research results.

    Uses an LLM to create or update a running summary based on the newest web research
    results, integrating them with any existing summary.

    Args:
        state: Current graph state containing research topic, running summary,
              and web research results
        config: Configuration for the runnable, including LLM provider settings

    Returns:
        Dictionary with state update, including running_summary key containing the updated summary
    """

    # Existing summary
    existing_summary = state.running_summary

    # Most recent web research
    most_recent_web_research = state.web_research_results[-1]

    # Build the human message
    if existing_summary:
        human_message_content = (
            f"<Existing Summary> \n {existing_summary} \n <Existing Summary>\n\n"
            f"<New Context> \n {most_recent_web_research} \n <New Context>"
            f"Update the Existing Summary with the New Context on this topic: \n <User Input> \n {state.research_topic} \n <User Input>\n\n"
        )
    else:
        human_message_content = (
            f"<Context> \n {most_recent_web_research} \n <Context>"
            f"Create a Summary using the Context on this topic: \n <User Input> \n {state.research_topic} \n <User Input>\n\n"
        )

    # Run the LLM
    configurable = Configuration.from_runnable_config(config)

    # For summarization, we don't need structured output, so always use regular mode
    if configurable.llm_provider == "lmstudio":
        llm = ChatLMStudio(
            base_url=configurable.lmstudio_base_url,
            model=configurable.local_llm,
            temperature=0,
        )
    else:  # Default to Ollama
        llm = ChatOllama(
            base_url=configurable.ollama_base_url,
            model=configurable.local_llm,
            temperature=0,
        )

    result = llm.invoke(
        [
            SystemMessage(content=summarizer_instructions),
            HumanMessage(content=human_message_content),
        ]
    )

    # Strip thinking tokens if configured
    running_summary = result.content
    if configurable.strip_thinking_tokens:
        running_summary = strip_thinking_tokens(running_summary)

    return {"running_summary": running_summary}


def reflect_on_summary(state: SummaryState, config: RunnableConfig):
    """LangGraph node that identifies knowledge gaps and generates follow-up queries.

    Analyzes the current summary to identify areas for further research and generates
    a new search query to address those gaps. Uses structured output to extract
    the follow-up query in JSON format.

    Args:
        state: Current graph state containing the running summary and research topic
        config: Configuration for the runnable, including LLM provider settings

    Returns:
        Dictionary with state update, including search_query key containing the generated follow-up query
    """

    # Generate a query
    configurable = Configuration.from_runnable_config(config)
    formatted_prompt = reflection_instructions.format(research_topic=state.research_topic)

    @tool
    class FollowUpQuery(BaseModel):
        """
        This tool is used to generate a follow-up query to address a knowledge gap.
        """

        follow_up_query: str = Field(description="Write a specific question to address this gap")
        knowledge_gap: str = Field(description="Describe what information is missing or needs clarification")

    messages = [
        SystemMessage(
            content=formatted_prompt
            + (tool_calling_reflection_instructions if configurable.use_tool_calling else json_mode_reflection_instructions)
        ),
        HumanMessage(
            content=f"Reflect on our existing knowledge: \n === \n {state.running_summary}, \n === \n And now identify a knowledge gap and generate a follow-up web search query:"
        ),
    ]

    return generate_search_query_with_structured_output(
        configurable=configurable,
        messages=messages,
        tool_class=FollowUpQuery,
        fallback_query=f"Tell me more about {state.research_topic}",
        tool_query_field="follow_up_query",
        json_query_field="follow_up_query",
    )


def finalize_summary(state: SummaryState):
    """LangGraph node that finalizes the research summary.

    Prepares the final output by deduplicating and formatting sources, then
    combining them with the running summary to create a well-structured
    research report with proper citations.

    Args:
        state: Current graph state containing the running summary and sources gathered

    Returns:
        Dictionary with state update, including running_summary key containing the formatted final summary with sources
    """

    # Deduplicate sources before joining
    seen_sources = set()
    unique_sources = []

    for source in state.sources_gathered:
        # Split the source into lines and process each individually
        for line in source.split("\n"):
            # Only process non-empty lines
            if line.strip() and line not in seen_sources:
                seen_sources.add(line)
                unique_sources.append(line)

    # Join the deduplicated sources
    all_sources = "\n".join(unique_sources)
    state.running_summary = f"## Summary\n{state.running_summary}\n\n ### Sources:\n{all_sources}"
    return {"running_summary": state.running_summary}


def route_research(state: SummaryState, config: RunnableConfig) -> Literal["finalize_summary", "web_research"]:
    """LangGraph routing function that determines the next step in the research flow.

    Controls the research loop by deciding whether to continue gathering information
    or to finalize the summary based on the configured maximum number of research loops.

    Args:
        state: Current graph state containing the research loop count
        config: Configuration for the runnable, including max_web_research_loops setting

    Returns:
        String literal indicating the next node to visit ("web_research" or "finalize_summary")
    """

    configurable = Configuration.from_runnable_config(config)
    if state.research_loop_count <= configurable.max_web_research_loops:
        return "web_research"
    else:
        return "finalize_summary"


# Add nodes and edges
builder = StateGraph(
    SummaryState,
    input=SummaryStateInput,
    output=SummaryStateOutput,
    config_schema=Configuration,
)
builder.add_node("generate_query", generate_query)
builder.add_node("web_research", web_research)
builder.add_node("summarize_sources", summarize_sources)
builder.add_node("reflect_on_summary", reflect_on_summary)
builder.add_node("finalize_summary", finalize_summary)

# Add edges
builder.add_edge(START, "generate_query")
builder.add_edge("generate_query", "web_research")
builder.add_edge("web_research", "summarize_sources")
builder.add_edge("summarize_sources", "reflect_on_summary")
builder.add_conditional_edges("reflect_on_summary", route_research)
builder.add_edge("finalize_summary", END)

graph = builder.compile()